# Charts for the check-in slides
Run the setup cell, then any chart. Screenshot each plot into the deck.

In [ ]:
from google.cloud import bigquery
import matplotlib.pyplot as plt

client = bigquery.Client(project="mcp-acc-055-dbg-p-7e23")
db = "`mcp-ss-data-p-5o6i`.vw_accelerate2605_core_v1"

# clean look for every chart
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["font.size"] = 13
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
BLUE = "#1f4e96"
RED  = "#c0392b"


### Chart 1 - our cohort funnel  (slides 2-3)

In [ ]:
# numbers from the claims notebook - update if those change
steps = ["All people", "Cancer patients", "Treated at Mayo", "Clear outcome"]
vals  = [3_500_000, 267_000, 76_000, 20_000]

fig, ax = plt.subplots()
ax.barh(steps[::-1], vals[::-1], color=BLUE)
for i, v in enumerate(vals[::-1]):
    ax.text(v, i, f"  {v:,}", va="center", fontsize=12)
ax.set_xscale("log")
ax.set_xlabel("number of people (log scale)")
ax.set_title("From all patients to our study cohort", fontweight="bold")
plt.tight_layout(); plt.show()


### Chart 2 - cancer diagnoses per year  (slide 2)

In [ ]:
sql = f"""
SELECT EXTRACT(YEAR FROM DATE_OF_DIAGNOSIS) AS year, COUNT(*) AS n
FROM {db}.FACT_CANCER_DATA_REPOSITORY
WHERE DATE_OF_DIAGNOSIS IS NOT NULL
GROUP BY 1 HAVING year BETWEEN 2005 AND 2024 ORDER BY 1
"""
d = client.query(sql).to_dataframe()

fig, ax = plt.subplots()
ax.bar(d["year"], d["n"], color=BLUE)
ax.set_ylabel("patients diagnosed")
ax.set_title("Cancer diagnoses per year at Mayo", fontweight="bold")
plt.tight_layout(); plt.show()


### Chart 3 - most patients arrive already diagnosed  (slide 3)

In [ ]:
sql = f"""
SELECT
  COUNTIF(DATE(DATE_OF_DIAGNOSIS) < DATE(DATE_FIRST_CONTACT)) AS referred_in,
  COUNTIF(DATE(DATE_OF_DIAGNOSIS) = DATE(DATE_FIRST_CONTACT)) AS same_day,
  COUNTIF(DATE(DATE_OF_DIAGNOSIS) > DATE(DATE_FIRST_CONTACT)) AS diagnosed_here
FROM {db}.FACT_CANCER_DATA_REPOSITORY
WHERE DATE_OF_DIAGNOSIS IS NOT NULL AND DATE_FIRST_CONTACT IS NOT NULL
"""
d = client.query(sql).to_dataframe().iloc[0]

labels = ["Diagnosed elsewhere,\nthen came to Mayo", "Same day", "First diagnosed\nat Mayo"]
vals   = [int(d.referred_in), int(d.same_day), int(d.diagnosed_here)]

fig, ax = plt.subplots()
ax.bar(labels, vals, color=[BLUE, "#7fa8d0", RED])
for i, v in enumerate(vals):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=12)
ax.set_ylabel("patients")
ax.set_title("Most cancer patients arrive already diagnosed", fontweight="bold")
plt.tight_layout(); plt.show()


### Chart 4 - how much data we can draw on  (slide 4)

In [ ]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM {db}.FACT_LAB_TEST)                  AS lab_results,
  (SELECT COUNT(*) FROM {db}.FACT_CLINICAL_DOCUMENTS)        AS clinical_notes,
  (SELECT COUNT(*) FROM {db}.FACT_RADIOLOGY_DICOM_INVENTORY) AS medical_images
"""
d = client.query(sql).to_dataframe().iloc[0]

labels = ["Lab results", "Clinical notes", "Medical images"]
vals   = [int(d.lab_results), int(d.clinical_notes), int(d.medical_images)]

fig, ax = plt.subplots()
ax.barh(labels[::-1], vals[::-1], color=BLUE)
for i, v in enumerate(vals[::-1]):
    txt = f"  {v/1e9:.1f} billion" if v >= 1e9 else f"  {v/1e6:.0f} million"
    ax.text(v, i, txt, va="center", fontsize=12)
ax.set_xscale("log")
ax.set_xlabel("records (log scale)")
ax.set_title("How much data we can draw on", fontweight="bold")
plt.tight_layout(); plt.show()


### Chart 5 - the outcomes we can measure  (slide 3)

In [ ]:
# numbers from the claims notebook - update if those change
labels = ["Cancer came back", "Stayed clear (2+ years)"]
vals   = [3400, 16800]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(vals, labels=labels, colors=[RED, BLUE], startangle=90,
       autopct=lambda p: f"{p:.0f}%\n({int(round(p/100*sum(vals))):,})",
       textprops={"fontsize": 12, "color": "white"}, pctdistance=0.7)
for t in ax.texts[:2]:
    t.set_color("black")
ax.set_title("Treatment outcomes we can measure", fontweight="bold")
plt.tight_layout(); plt.show()
